## Human feedback in the Loop

In [2]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm_model = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [12]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition

from langgraph.types import Command,interrupt

class State(TypedDict):
    messages:Annotated[list,add_messages]
    

graph_builder = StateGraph(State)

@tool
def human_assistance(query:str)-> str:
    """Request human assistance for the given query."""
    human_response = interrupt({"query": query})
    return human_response["data"]


web_search_tool = TavilySearch(max_results=3)

tools = [web_search_tool, human_assistance]

# tools bindings
llm_model.bind_tools(tools)

# node definitions
def chatbot(state:State)-> str:
    message = llm_model.invoke(state["messages"])
    
    return {"messages":[message]}

# adding nodes to graph
graph_builder.add_node("chatbot",chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)
# adding edges to graph
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("tools", END)




In [ ]:
memory = MemorySaver()
compiled_graph = graph_builder.compile(checkpointer=memory)

In [18]:
from IPython.display import display,Image

try:
    display(Image(compiled_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization failed:", e)

Graph visualization failed: name 'compiled_graph' is not defined


In [ ]:
# 1:55